In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import difflib
import re

In [7]:
treadmill = pd.read_excel("../../data/cleaned/acceptances/treadmill.xlsx")
runners = pd.read_excel("../../data/cleaned/results/runners.xlsx")

In [3]:
# 1. Get unique names and clean them to find potential duplicates
unique_names = treadmill.horse_name.dropna().unique()

# 2. Look for close matches
similar_pairs = []
for i, name1 in enumerate(unique_names):
    # Get close matches from the rest of the list
    matches = difflib.get_close_matches(name1, unique_names[i+1:], n=3, cutoff=0.8)
    for match in matches:
        similar_pairs.append((name1, match))

# 3. View the results clearly
df_typos = pd.DataFrame(similar_pairs, columns=['Name A', 'Name B'])
df_typos

,Name A,Name B
0,ARCADIA,ARKADIAN


In [8]:
unique_names = treadmill.horse_name.dropna().unique()

# Create a DataFrame of unique names
df_names = pd.DataFrame({'original_name': unique_names})

# Create a 'clean' key: uppercase, no spaces, no punctuation
df_names['clean_key'] = df_names['original_name'].apply(
    lambda x: re.sub(r'[^A-Z0-9]', '', str(x).upper())
)

# Find clean keys that appear more than once (meaning they have variations)
duplicates = df_names[df_names.duplicated(subset=['clean_key'], keep=False)]

# Sort them so variations sit next to each other
duplicates.sort_values(by='clean_key')

,original_name,clean_key


In [12]:
# =========================
# 1. NORMALIZE
# =========================
def normalize(df):
    df = df.copy()
    df['meet_date'] = pd.to_datetime(df['meet_date'])
    df['horse_name'] = df['horse_name'].astype(str).str.strip().str.upper()
    return df

runners = normalize(runners)
treadmill = normalize(treadmill)

# =========================
# 2. BUILD RUNNER MAP
# =========================
runner_map = runners[['meet_date', 'horse_name', 'race_no']].drop_duplicates()
runner_grouped = runner_map.groupby('meet_date')

# =========================
# 3. INITIAL MERGE
# =========================
merged = treadmill.merge(
    runner_map,
    on=['meet_date', 'horse_name'],
    how='left',
    indicator=True,
    suffixes=('_old', '_true')
)

# unified race column
merged['race_no_final'] = merged['race_no_true']

exact_matched = merged[merged['_merge'] == 'both'].copy()
unmatched = merged[merged['_merge'] == 'left_only'].copy()

# =========================
# 4. PARTIAL MATCH
# =========================
resolved_rows = []
still_unmatched = []

for _, row in unmatched.iterrows():
    date = row['meet_date']
    name = row['horse_name']

    if date not in runner_grouped.groups:
        still_unmatched.append(row)
        continue

    candidates = runner_grouped.get_group(date)

    matches = candidates[
        candidates['horse_name'].apply(lambda x: x.startswith(name))
    ]

    if len(matches) >= 1:
        best = matches.loc[matches['horse_name'].str.len().idxmax()]

        row['race_no_final'] = best['race_no']
        row['horse_name'] = best['horse_name']

        resolved_rows.append(row)
    else:
        still_unmatched.append(row)

partial_matched = pd.DataFrame(resolved_rows)
edge_cases = pd.DataFrame(still_unmatched)

# =========================
# 5. COMBINE
# =========================
clean = pd.concat([exact_matched, partial_matched, edge_cases], ignore_index=True)

# =========================
# 6. FINAL RACE_NO FIX
# =========================
clean['race_no'] = clean['race_no_final'].fillna(clean['race_no_old'])

# =========================
# 7. FLAG
# =========================
clean['is_current_runner'] = clean['race_no_final'].notna()

# =========================
# 8. CLEANUP
# =========================
clean = clean.drop(
    columns=['_merge', 'race_no_old', 'race_no_true', 'race_no_final'],
    errors='ignore'
)

clean = clean.sort_values(by=['meet_date', 'race_no']).reset_index(drop=True)

clean['meet_date'] = clean['meet_date'].dt.strftime('%Y-%m-%d')
clean['date'] = pd.to_datetime(clean['date']).dt.strftime('%Y-%m-%d')

clean.to_excel("treadmill.xlsx", index=False)